# Statistics

This notebook fetches and displays tabular basketball player statistics from the Basketball Stats Vlaanderen website for a specific player and season.

In [ ]:
club = "https://app.basketballstatsvlaanderen.be/clubs/BVBL1037/BVBL1037J16%20%201#stats"

In [19]:
clubmatchesurl = "https://vblcb.wisseq.eu/VBLCB_WebService/data/OrgMatchesByGuid?issguid=BVBL1037"

## FUNCTIONS  fetch_player_summary(url)  and fetch_player(url)


In [ ]:
def fetch_player_summary(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    fields_to_keep = [
        'Name', 'Number', 'FunctionLetter', 'Starter',
        'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
        'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus',
        'GameTeam_GameDate', 'GameTeam_GameResult', 'GameTeam_GameGuid',
        'GameTeam_HomeTeamName', 'GameTeam_AwayTeamName'
    ]

    filtered_games_data = [
        {k: game[k] for k in fields_to_keep if k in game}
        for game in games_data
    ]

    df_games = pd.DataFrame(filtered_games_data)

    fields_to_average = [
        'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
        'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus'
    ]

    summary_data = []
    for name, group in df_games.groupby('Name'):
        entry = {
            'Name': name,
            'Number': group['Number'].iloc[0] if 'Number' in group.columns else None,
            'Starter %': group['Starter'].mean() * 100 if 'Starter' in group.columns else None
        }
        for field in fields_to_average:
            filtered_group = group[group['TotalMinutes'] > 0]
            entry[field + ' Avg'] = filtered_group[field].mean() if field in filtered_group.columns else None
        summary_data.append(entry)

    summary_df = pd.DataFrame(summary_data)
    return summary_df


In [19]:
def fetch_player(url):
    import requests
    import re
    from bs4 import BeautifulSoup
    import json
    import pandas as pd

    response = requests.get(url)
    html_doc = response.text

    soup = BeautifulSoup(html_doc, 'html.parser')
    games_data = []

    for script in soup.find_all('script'):
        if script.string and "var games = [" in script.string:
            match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
            if match:
                games_str = match.group(0).replace('var games =', '').strip(' ;')
                try:
                    games_data = json.loads(games_str)
                    break
                except json.JSONDecodeError as e:
                    print(f"Error decoding JSON: {e}")

    fields_to_keep = [
        'Name', 'Number', 'FunctionLetter', 'Starter',
        'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
        'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus',
        'GameTeam_GameDate', 'GameTeam_GameResult', 'GameTeam_GameGuid',
        'GameTeam_HomeTeamName', 'GameTeam_AwayTeamName'
    ]

    filtered_games_data = [
        {k: game[k] for k in fields_to_keep if k in game}
        for game in games_data
    ]

    df_games = pd.DataFrame(filtered_games_data)

    # fields_to_average = [
    #     'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
    #     'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus'
    # ]

    # summary_data = []
    # for name, group in df_games.groupby('Name'):
    #     entry = {
    #         'Name': name,
    #         'Number': group['Number'].iloc[0] if 'Number' in group.columns else None,
    #         'Starter %': group['Starter'].mean() * 100 if 'Starter' in group.columns else None
    #     }
    #     for field in fields_to_average:
    #         filtered_group = group[group['TotalMinutes'] > 0]
    #         entry[field + ' Avg'] = filtered_group[field].mean() if field in filtered_group.columns else None
    #     summary_data.append(entry)

    # summary_df = pd.DataFrame(summary_data)
    return df_games


# execute function

In [21]:
# Fetch Player Data from Webpage
url = "https://app.basketballstatsvlaanderen.be/players/BVBL744354"
# url = "https://app.basketballstatsvlaanderen.be/players/BVBL711026"

data = fetch_player_summary(url)
data2 = fetch_player(url)
data2[data2['Number'] == 8].sort_values(by='TotalScore', ascending=False)

,Name,Number,FunctionLetter,Starter,TotalMinutes,NormalizedMinutes,FreeThrows,FieldGoals,ThreePointers,TotalScore,PlusMinus,Faults,Plus,Minus
3,Vic Huysman,8,S,True,33,37,6,20,0,26,3,4,61,-58
0,Vic Huysman,8,S,True,34,38,5,20,0,25,-58,1,39,-97
6,Vic Huysman,8,S,True,22,22,5,16,3,24,-34,3,49,-83


In [199]:
# Read the PLOEG.HTML file
with open(r'C:\Users\StijnHuysman\OneDrive - mateco cloud\GITHUB REPOS\HAANTJES\PLOEG.HTML', 'r', encoding='utf-8') as file:
    ploeg_html = file.read()

# Parse the HTML content
soup = BeautifulSoup(ploeg_html, 'html.parser')

# Extract player names and their corresponding href links
players = []
for a_tag in soup.find_all('a', href=True):
    if 'player' in a_tag['href']:  # Filter only player links
        players.append({'NAME': a_tag.text.strip(), 'HREF': a_tag['href']})
        # Extract SPELERID from the HREF field
        for player in players:
            player['SPELERID'] = player['HREF'].split('/')[-1]
# Create a DataFrame to display the players
# players_df = pd.DataFrame(players, columns=['NAME', 'HREF'])
# Convert the players list to a DataFrame
players_df = pd.DataFrame(players, columns=['NAME', 'HREF', 'SPELERID'])
# Remove duplicate rows based on 'NAME' and 'SPELERID' columns
players_df = players_df.drop_duplicates(subset=['NAME', 'SPELERID'])
# Display the DataFrame
players_df['HREF']
players_df['SPELERID']
# Create a new DataFrame with player summaries using fetch_player_summary for each SPELERID
player_summaries = []
for _, row in players_df.iterrows():
    player_url = f"https://app.basketballstatsvlaanderen.be/players/{row['SPELERID']}?season=2425"
    summary_df = fetch_player_summary(player_url)
    summary_df['SPELERID'] = row['SPELERID']
    summary_df['NAME'] = row['NAME']
    player_summaries.append(summary_df)

all_players_summary_df = pd.concat(player_summaries, ignore_index=True)
all_players_summary_df
for index, row in players_df.iterrows():
    print(f"Name: {row['NAME']}, SPELERID: {row['SPELERID']}")




ValueError: No objects to concatenate

In [184]:
all_players_summary_df

,Name,Number,Starter %,TotalMinutes Avg,NormalizedMinutes Avg,FreeThrows Avg,FieldGoals Avg,ThreePointers Avg,TotalScore Avg,PlusMinus Avg,Faults Avg,Plus Avg,Minus Avg,SPELERID,NAME
0,Gust Ottevaere,11,0.000000,11.846154,13.461538,0.307692,3.076923,0.230769,3.615385,8.692308,1.307692,25.846154,-17.153846,BVBL750587,Gust Ottevaere
1,Jasper Espeel,11,83.333333,21.777778,25.444444,1.777778,5.777778,0.833333,8.388889,0.333333,1.666667,38.055556,-37.722222,BVBL749137,Jasper Espeel
2,Jonah Cattoir,5,69.444444,22.361111,27.388889,2.361111,11.444444,0.000000,13.805556,3.194444,2.055556,43.250000,-40.055556,BVBL762639,Jonah Cattoir
3,Marcel Heerman,12,17.857143,15.222222,17.481481,0.407407,5.777778,0.000000,6.185185,17.444444,1.666667,37.296296,-19.851852,BVBL668586,Marcel Heerman
4,Mathis Cromheeke,14,85.714286,22.000000,25.714286,0.571429,3.142857,0.285714,4.000000,-2.000000,0.904762,40.571429,-42.571429,BVBL715872,Mathis Cromheeke
5,Niels De Coussemaker,6,9.523810,11.941176,13.823529,0.352941,0.705882,0.000000,1.058824,12.058824,1.000000,25.529412,-13.470588,BVBL708695,Niels De Coussemaker
6,Tibo Despriet,12,95.833333,22.166667,26.541667,1.125000,14.333333,0.250000,15.708333,28.791667,1.833333,62.583333,-33.791667,BVBL738959,Tibo Despriet
7,Torre Beeckman,15,95.833333,21.208333,25.166667,1.333333,8.916667,1.000000,11.250000,23.416667,0.666667,56.291667,-32.875000,BVBL720212,Torre Beeckman
8,Vic Huysman,8,83.870968,21.774194,25.967742,1.516129,5.935484,0.000000,7.451613,22.161290,1.548387,55.870968,-33.709677,BVBL744354,Vic Huysman
9,Viktor L'Hommelet,5,91.304348,21.347826,25.565217,1.043478,14.869565,1.043478,16.956522,25.521739,1.434783,57.913043,-32.391304,BVBL714059,Viktor L'Hommelet


In [ ]:

# Fetch Player Data from Webpage
url = "https://app.basketballstatsvlaanderen.be/players/BVBL744354?season=2425"

response = requests.get(url)
html_doc = response.text

soup = BeautifulSoup(html_doc, 'html.parser')
games_data = []

for script in soup.find_all('script'):
    if script.string and "var games = [" in script.string:
        # Use a regex to find the content of the array
        match = re.search(r"var games = \[.*?\];", script.string, re.DOTALL)
        if match:
            # Extract the string content
            games_str = match.group(0).replace('var games =', '').strip(' ;')
            try:
                # Use json.loads to parse the array string into a Python list
                games_data = json.loads(games_str)
                break  # Stop once we've found and extracted the data
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")

fields_to_keep = [
    # 'id', 'GameTeamId', 'Guid', 
    'Name', 'Number', 'FunctionLetter', 'Starter',
    'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
    'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus' ,  
    # 'GameTeam_GameId', 
    'GameTeam_GameDate', 'GameTeam_GameResult', 'GameTeam_GameGuid',
    'GameTeam_HomeTeamName', 'GameTeam_AwayTeamName'
]

filtered_games_data = [
    {k: game[k] for k in fields_to_keep if k in game}
    for game in games_data
]

filtered_games_data
# Create a DataFrame from filtered_games_data
df_games = pd.DataFrame(filtered_games_data)



# Calculate the average for the specified fields
fields_to_average = [
    'TotalMinutes', 'NormalizedMinutes', 'FreeThrows', 'FieldGoals', 'ThreePointers',
    'TotalScore', 'PlusMinus', 'Faults', 'Plus', 'Minus'
]
df_games_grouped = df_games.groupby('Name')[fields_to_average].mean()
averages = {field: df_games[field].mean() for field in fields_to_average if field in df_games.columns}
# Prepare the summary DataFrame
summary_data = []

for name, group in df_games.groupby('Name'):
    entry = {
        'Name': name,
        'Number': group['Number'].iloc[0] if 'Number' in group.columns else None,
        'Starter %': group['Starter'].mean() * 100 if 'Starter' in group.columns else None
    }
    for field in fields_to_average:
        filtered_group = group[group['TotalMinutes'] > 0]
        entry[field + ' Avg'] = filtered_group[field].mean() if field in filtered_group.columns else None
    summary_data.append(entry)

summary_df = pd.DataFrame(summary_data)
summary_df
